In [ ]:
# Fixed Group-Aware Data Partitioning

# This notebook creates the fixed train, validation, and test partitions used throughout the experiments.
# Splitting is performed at leakage-group level so that related vulnerability records cannot appear across different partitions.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit
)

from src.config import (
    LABEL_COL,
    GROUP_COL,
    ID_COL,
    RU_PARTITIONS_FILE,
    NONRU_PARTITIONS_FILE,
    RESULTS_TABLES_DIR,
)

from src.data import load_leakage_data

ru_df, nonru_df = load_leakage_data()

RANDOM_STATE = 42
RU_TEST_SIZE = 0.15
RU_VAL_SIZE = 0.15

In [ ]:
## Group-Aware Split Selection

# A repeated group-aware splitting procedure is defined.
# Candidate splits are evaluated according to target size and CWE-distribution similarity
# while requiring all 30 classes to remain represented.

In [3]:
def find_best_group_split(
    df,
    holdout_fraction,
    random_state,
    n_trials=2000
):

    full_distribution = (
        df[LABEL_COL]
        .value_counts(normalize=True)
        .sort_index()
    )

    all_classes = set(
        df[LABEL_COL].unique()
    )

    splitter = GroupShuffleSplit(
        n_splits=n_trials,
        test_size=holdout_fraction,
        random_state=random_state
    )

    best = None
    best_score = np.inf


    for train_idx, holdout_idx in (
        splitter.split(
            df,
            y=df[LABEL_COL],
            groups=df[GROUP_COL]
        )
    ):

        train = df.iloc[train_idx]
        holdout = df.iloc[holdout_idx]

        if set(train[LABEL_COL]) != all_classes:
            continue

        if set(holdout[LABEL_COL]) != all_classes:
            continue


        actual_fraction = (
            len(holdout)
            /
            len(df)
        )

        holdout_distribution = (
            holdout[LABEL_COL]
            .value_counts(normalize=True)
            .reindex(
                full_distribution.index,
                fill_value=0
            )
        )

        difference = (
            holdout_distribution
            -
            full_distribution
        ).abs()


        score = (
            2 * abs(
                actual_fraction
                -
                holdout_fraction
            )
            +
            difference.mean()
            +
            0.25 * difference.max()
        )


        if score < best_score:

            best_score = score

            best = (
                train_idx,
                holdout_idx
            )


    if best is None:
        raise RuntimeError(
            "No suitable split found."
        )

    return (
        best[0],
        best[1],
        best_score
    )

In [ ]:
## Create the Fixed Russian Test Set

# The Russian dataset is divided into development and test data using leakage groups rather than individual records.
# The selected test split is approximately 15% of the Russian modelling dataset.

In [4]:
ru_df = (
    ru_df
    .copy()
    .reset_index(drop=True)
)


ru_dev_idx, ru_test_idx, test_score = (
    find_best_group_split(
        ru_df,
        holdout_fraction=RU_TEST_SIZE,
        random_state=RANDOM_STATE
    )
)


ru_dev = ru_df.iloc[ru_dev_idx].copy()
ru_test = ru_df.iloc[ru_test_idx].copy()


print("RU development:", len(ru_dev))
print("RU test:", len(ru_test))
print("Test classes:", ru_test[LABEL_COL].nunique())
print("Split score:", test_score)

RU development: 12280
RU test: 2171
Test classes: 30
Split score: 0.0033355803736156036


In [ ]:
## Split Russian Development Data

# The remaining Russian development data are divided into training and
# validation partitions while preserving group isolation and class coverage.

In [5]:
VAL_FRACTION_OF_DEV = (
    RU_VAL_SIZE
    /
    (1 - RU_TEST_SIZE)
)


ru_train_idx, ru_val_idx, val_score = (
    find_best_group_split(
        ru_dev,
        holdout_fraction=VAL_FRACTION_OF_DEV,
        random_state=RANDOM_STATE + 1
    )
)


ru_train = ru_dev.iloc[ru_train_idx].copy()
ru_val = ru_dev.iloc[ru_val_idx].copy()


ru_df["split"] = pd.NA

ru_df.loc[
    ru_df[ID_COL].isin(ru_train[ID_COL]),
    "split"
] = "train"

ru_df.loc[
    ru_df[ID_COL].isin(ru_val[ID_COL]),
    "split"
] = "val"

ru_df.loc[
    ru_df[ID_COL].isin(ru_test[ID_COL]),
    "split"
] = "test"


print(
    ru_df["split"]
    .value_counts()
)

split
train    10114
test      2171
val       2166
Name: count, dtype: int64


In [ ]:
## Verify Russian Group Isolation

# The leakage groups assigned to Russian train, validation, and test partitions are
# checked to confirm that no group occurs in more than one partition.

In [7]:
ru_train_groups = set(
    ru_train[GROUP_COL]
)

ru_val_groups = set(
    ru_val[GROUP_COL]
)

ru_test_groups = set(
    ru_test[GROUP_COL]
)


assert not (
    ru_train_groups
    &
    ru_val_groups
)

assert not (
    ru_train_groups
    &
    ru_test_groups
)

assert not (
    ru_val_groups
    &
    ru_test_groups
)


print(
    "RU train/validation/test "
    "are group-isolated."
)

RU train/validation/test are group-isolated.


In [ ]:
## Align Non-Russian Partitions with the Russian Split

# Non-Russian records linked to the fixed Russian test set are excluded.
# Cross-domain leakage groups associated with Russian training or validation data
# are assigned consistently to the corresponding non-Russian partition.

In [8]:
nonru_df = (
    nonru_df
    .copy()
    .reset_index(drop=True)
)

nonru_df["split"] = pd.NA
nonru_df["exclusion_reason"] = pd.NA


overlap_test = (
    nonru_df[GROUP_COL]
    .isin(ru_test_groups)
)

nonru_df.loc[
    overlap_test,
    "split"
] = "excluded"

nonru_df.loc[
    overlap_test,
    "exclusion_reason"
] = "leakage_group_overlaps_RU_test"


forced_train = (
    nonru_df["split"].isna()
    &
    nonru_df[GROUP_COL].isin(
        ru_train_groups
    )
)

forced_val = (
    nonru_df["split"].isna()
    &
    nonru_df[GROUP_COL].isin(
        ru_val_groups
    )
)


nonru_df.loc[
    forced_train,
    "split"
] = "train"

nonru_df.loc[
    forced_val,
    "split"
] = "val"


print(
    "Excluded:",
    overlap_test.sum()
)

print(
    "Forced train:",
    forced_train.sum()
)

print(
    "Forced validation:",
    forced_val.sum()
)

Excluded: 35
Forced train: 632
Forced validation: 66


In [ ]:
## Partition the Remaining Non-Russian Data

# Records not constrained by cross-domain leakage groups are split into training and validation data
# while maintaining the intended validation proportion and CWE coverage.

In [10]:
remaining = (
    nonru_df[
        nonru_df["split"].isna()
    ]
    .copy()
)


eligible_n = (
    nonru_df["split"]
    .ne("excluded")
    .sum()
)

forced_val_n = (
    nonru_df["split"]
    .eq("val")
    .sum()
)

target_val_n = round(
    eligible_n * 0.15
)

remaining_val_fraction = (
    (
        target_val_n
        -
        forced_val_n
    )
    /
    len(remaining)
)


nr_train_idx, nr_val_idx, _ = (
    find_best_group_split(
        remaining,
        holdout_fraction=remaining_val_fraction,
        random_state=RANDOM_STATE + 2
    )
)


nr_train = (
    remaining
    .iloc[nr_train_idx]
)

nr_val = (
    remaining
    .iloc[nr_val_idx]
)


nonru_df.loc[
    nonru_df[ID_COL].isin(
        nr_train[ID_COL]
    ),
    "split"
] = "train"

nonru_df.loc[
    nonru_df[ID_COL].isin(
        nr_val[ID_COL]
    ),
    "split"
] = "val"


assert nonru_df["split"].notna().all()

print(
    nonru_df["split"]
    .value_counts()
)

split
train       35391
val          6237
excluded       35
Name: count, dtype: int64


In [ ]:
## Validate the Final Partition Structure

# Group isolation is checked across the non-Russian partitions and against the Russian test set.
# Each experimental partition is also verified to contain all 30 CWE classes.

In [11]:
nonru_train_groups = set(
    nonru_df.loc[
        nonru_df["split"] == "train",
        GROUP_COL
    ]
)

nonru_val_groups = set(
    nonru_df.loc[
        nonru_df["split"] == "val",
        GROUP_COL
    ]
)


assert not (
    nonru_train_groups
    &
    nonru_val_groups
)

assert not (
    nonru_train_groups
    &
    ru_test_groups
)

assert not (
    nonru_val_groups
    &
    ru_test_groups
)


partitions = {
    "RU train":
        ru_df[
            ru_df["split"] == "train"
        ],

    "RU validation":
        ru_df[
            ru_df["split"] == "val"
        ],

    "RU test":
        ru_df[
            ru_df["split"] == "test"
        ],

    "Non-RU train":
        nonru_df[
            nonru_df["split"] == "train"
        ],

    "Non-RU validation":
        nonru_df[
            nonru_df["split"] == "val"
        ],
}


for name, dataframe in partitions.items():

    print(
        name,
        len(dataframe),
        dataframe[LABEL_COL].nunique()
    )

    assert (
        dataframe[LABEL_COL]
        .nunique()
        == 30
    )

RU train 10114 30
RU validation 2166 30
RU test 2171 30
Non-RU train 35391 30
Non-RU validation 6237 30


In [ ]:
## Reproduce the Fixed Split Counts

# The final record counts are checked against the fixed partition sizes used throughout the dissertation experiments.

In [13]:
assert (
    ru_df["split"] == "train"
).sum() == 10114

assert (
    ru_df["split"] == "val"
).sum() == 2166

assert (
    ru_df["split"] == "test"
).sum() == 2171

assert (
    nonru_df["split"] == "train"
).sum() == 35391

assert (
    nonru_df["split"] == "val"
).sum() == 6237

assert (
    nonru_df["split"] == "excluded"
).sum() == 35

print("Fixed partition counts reproduced.")

Fixed partition counts reproduced.


In [ ]:
## Export the Fixed Partitions

# The Russian and non-Russian partitioned datasets and a concise partition summary
# are saved for use in all subsequent preprocessing and modelling stages.

In [16]:
partition_summary = pd.DataFrame({

    "partition":
        list(partitions.keys())
        +
        ["Non-RU excluded"],

    "records":
        [
            len(df)
            for df in partitions.values()
        ]
        +
        [
            (
                nonru_df["split"]
                == "excluded"
            ).sum()
        ]
})


ru_df.to_excel(
    RU_PARTITIONS_FILE,
    index=False
)

nonru_df.to_excel(
    NONRU_PARTITIONS_FILE,
    index=False
)

partition_summary.to_csv(
    RESULTS_TABLES_DIR
    / "fixed_partition_summary.csv",
    index=False
)


print("Fixed partitions saved.")

Fixed partitions saved.
